In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [2]:
import duckdb
print("duckdb ok")

duckdb ok


In [3]:
import duckdb

con = duckdb.connect()

path = r"C:\Users\Fausto\Downloads\train_rsample_100k.parquet"

df = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
""").df()

print(df.shape)
print(df.head())

(100000, 10)
   datetime ui_language selected_template              eligible_templates  \
0  9.283380          fr                 L     [K, H, G, E, B, J, L, F, D]   
1  9.623044          es                 D     [K, H, G, E, B, J, L, F, D]   
2  5.172986          de                 K  [G, E, B, A, K, H, J, L, F, D]   
3  6.269595          en                 G     [G, E, B, K, H, J, L, F, D]   
4  0.913009          es                 L     [G, E, B, K, H, J, L, F, D]   

                                             history  history_length  \
0  [{'template': 'J', 'n_days': 5.882936954498291...               3   
1  [{'template': 'A', 'n_days': 23.98947525024414...              11   
2  [{'template': 'E', 'n_days': 4.508449554443359...               3   
3  [{'template': 'K', 'n_days': 8.594528198242188...               7   
4  [{'template': 'L', 'n_days': 0.9999991059303284}]               1   

   n_eligible  time_of_day  session_end_completed   hour_utc  
0           9     9.283380  

1) How big is this dataset?

In [4]:
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{path}')
""").df()

,n_rows
0,100000


2) What is the global sucess rate (reward)?

In [5]:
con.execute(f"""
SELECT AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
""").df()

,reward_rate
0,0.14704


In [6]:
con.execute(f"""
SELECT
  regexp_extract(filename, 'train-part-\\d+', 0) AS part,
  COUNT(*) AS n,
  AVG(CAST(session_end_completed AS INT)) AS reward_rate
FROM read_parquet('{path}', filename=true)
GROUP BY 1
ORDER BY 1
""").df()

,part,n,reward_rate
0,,100000,0.14704


3) How often occurs every template + how good does it work?

In [7]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,C,3102,0.413282
1,A,4053,0.272144
2,K,10395,0.136893
3,L,10267,0.134606
4,D,10303,0.134039
5,G,10332,0.133662
6,H,10239,0.133411
7,E,10422,0.131549
8,B,10342,0.131213
9,F,10309,0.129887


4) Does it differ per language?

In [8]:
con.execute(f"""
SELECT
  ui_language,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY ui_language
ORDER BY n DESC
""").df()

,ui_language,n,reward_rate
0,en,40809,0.165650
1,es,24250,0.124866
2,pt,9502,0.128184
3,ru,4512,0.143839
4,fr,3859,0.156776
5,de,2871,0.199930
6,zs,1866,0.094855
7,ar,1825,0.058630
8,it,1753,0.143183
9,vi,1573,0.140496


5) Important for bandit: how many templates are eligible per event?

In [9]:
con.execute(f"""
SELECT
  AVG(LEN(eligible_templates)) AS avg_eligible,
  MIN(LEN(eligible_templates)) AS min_eligible,
  MAX(LEN(eligible_templates)) AS max_eligible
FROM read_parquet('{path}')
""").df()

,avg_eligible,min_eligible,max_eligible
0,9.14069,1,10


How often is C available?

In [10]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS c_available
FROM read_parquet('{path}')
""").df()

,total,c_available
0,100000,3102.0


What if we always choose C when C is available, 
and otherwise the best of the rest?

So firstly we must know:
What is the best template without C?

In [11]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' NOT IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,A,4053,0.272144
1,K,10395,0.136893
2,L,10267,0.134606
3,D,10303,0.134039
4,G,10332,0.133662
5,H,10239,0.133411
6,E,10422,0.131549
7,B,10342,0.131213
8,F,10309,0.129887
9,J,10236,0.128859


There is a clear hierarchy:
C-> best overall(41.2%)
A -> best if C is not available (26.9%)
Rest -> all around 13%

This is no subtle difference.
A strong rule-based policy would be:
If C available -> choose C
Else -> choose A

1) How often would our new policy choose C vs A?

In [12]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_C,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_A,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' NOT IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_neither
FROM read_parquet('{path}')
""").df()

,total,policy_choose_C,policy_choose_A,policy_choose_neither
0,100000,3102.0,40530.0,56368.0


2) Off-policy estimate: reward of policy “C else A”

In [13]:
con.execute(f"""
WITH data AS (
  SELECT
    session_end_completed,
    selected_template,
    CASE 
      WHEN 'C' IN eligible_templates THEN 'C'
      ELSE 'A'
    END AS policy_template
  FROM read_parquet('{path}')
)
SELECT
  COUNT(*) AS total_events,
  SUM(CASE WHEN selected_template = policy_template THEN 1 ELSE 0 END) AS matched_events,
  AVG(CASE WHEN selected_template = policy_template THEN CASE WHEN session_end_completed THEN 1 ELSE 0 END END) AS estimated_policy_reward
FROM data
""").df()

,total_events,matched_events,estimated_policy_reward
0,100000,7155.0,0.333333


What do we see?
Total events
87,665,839
Matched events
5,996,554 (useful for evaluation)
Estimated policy reward
0.329 (32.9%)
Comparison with current logging policy(=In the historical data, Duolingo selected a template at random from the eligible pool, with equal probability for each template. That random decision process is what generated the dataset — that’s the logging policy)
Logging policy reward was:
14.4%
Our simple new rule:
If C available -> C
Else -> A
32.9%

A matched event is:
An event where the logging policy chose by chance exactly the same template as your new policy would have chosen.
Why do we only use matched events?

!!!Because we only know the reward of what was actually shown!!!

For example:
Suppose:
Eligible = {C, D, E}
Logging chose D
Our new policy would have chosen C
Then we know:
Reward of D -> yes
Reward of C -> no (counterfactual unknown)
So we can't use that event to evaluate C.

This compares C vs A in exactly the same context/set/pool
It removes a big part of the selection bias.

In [14]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' IN eligible_templates
  AND 'A' IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate


What does this mean in terms of content?

!!!C and A apparently never occur in the same eligible pool!!!

So:
!!!C is shown in a completely different segment than A,
this confirms selection bias!!!
You can't compare C and A within the same context.

The rule:
If C available -> C
Else -> A
actually works because:
C-segment = high engagement users
A-segment = other users
So your policy is actually:
If user in C-segment -> choose C
Otherwise -> choose A
But that segment difference is already incorporated in the data.

1) Which eligible pools occur the most?

In [15]:
con.execute(f"""
SELECT
  eligible_templates,
  COUNT(*) AS n
FROM read_parquet('{path}')
GROUP BY eligible_templates
ORDER BY n DESC
LIMIT 20
""").df()

,eligible_templates,n
0,"[G, E, B, K, H, J, L, F, D]",40698
1,"[G, E, B, A, K, H, J, L, F, D]",31202
2,"[K, H, G, E, B, J, L, F, D]",15611
3,"[K, H, G, E, B, J, L, F, D, A]",9152
4,[C],3102
5,"[A, K, H]",129
6,"[K, H]",59
7,"[K, H, A]",47


2) Within every pool: reward per selected_template (and top/best template per pool)

In [16]:
con.execute(f"""
WITH stats AS (
  SELECT
    eligible_templates,
    selected_template,
    COUNT(*) AS n,
    AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
  FROM read_parquet('{path}')
  GROUP BY eligible_templates, selected_template
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY eligible_templates
      ORDER BY reward_rate DESC
    ) AS rnk
  FROM stats
  WHERE n >= 5000
)
SELECT
  eligible_templates,
  selected_template AS best_template,
  n,
  reward_rate
FROM ranked
WHERE rnk = 1
ORDER BY n DESC
LIMIT 50
""").df()

,eligible_templates,best_template,n,reward_rate


What do we see here?
There are a few big eligible pools (million events).
The best template is not always C within these pools.
In some pools wins L.
In little pools (such as [A, K, H]) wins A with a high reward rate.
C is actually in its own pool= [C].
That means:
Templates live in different worlds.
Reward rate is strongly dependent of the eligible pool.

Logistic Model:

STEP1 : load the data (fast + safe) with DuckDB in Python

In [17]:
import duckdb
import pandas as pd

# 1) Connect
con = duckdb.connect()

# 2) Zet hier je pad naar de parquet files (training of test)
# Voorbeeld: "data/training/*.parquet" of "/path/to/training/*.parquet"
PARQUET_GLOB = r"C:\Users\Fausto\Downloads\train_rsample_100k.parquet"

# 3) Maak een view (handig om later queries te hergebruiken)
con.execute(f"""
CREATE OR REPLACE VIEW notif AS
SELECT *
FROM read_parquet('{PARQUET_GLOB}');
""")

# 4) Snelle sanity checks
print("Row count (approx query):")
print(con.execute("SELECT COUNT(*) AS n FROM notif").fetchdf())

print("\nColumns + types:")
print(con.execute("DESCRIBE notif").fetchdf())

print("\nOverall reward rate:")
print(con.execute("""
SELECT AVG(CAST(session_end_completed AS INTEGER)) AS reward_rate
FROM notif
""").fetchdf())

print("\nExample rows:")
print(con.execute("""
SELECT datetime, ui_language, eligible_templates, history, selected_template, session_end_completed
FROM notif
LIMIT 3
""").fetchdf())

Row count (approx query):
        n
0  100000

Columns + types:
             column_name                                 column_type null  \
0               datetime                                      DOUBLE  YES   
1            ui_language                                     VARCHAR  YES   
2      selected_template                                     VARCHAR  YES   
3     eligible_templates                                   VARCHAR[]  YES   
4                history  STRUCT("template" VARCHAR, n_days FLOAT)[]  YES   
5         history_length                                    UINTEGER  YES   
6             n_eligible                                    UINTEGER  YES   
7            time_of_day                                      DOUBLE  YES   
8  session_end_completed                                     BOOLEAN  YES   
9               hour_utc                                      DOUBLE  YES   

    key default extra  
0  None    None  None  
1  None    None  None  
2  None    None 

Step 2: drawing a training sample + prepare features
We want:
Only real decision points (len(eligible_templates) >= 2)
A sample of ±1 miljoen rows (for speed)

In [18]:
query = """
SELECT
    ui_language,
    selected_template,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,
    (
      SELECT min(h.n_days)
      FROM unnest(history) AS u(h)
      WHERE h.template = selected_template
    ) AS days_since_last_notification,
    history_length,
    time_of_day
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 1000000 ROWS
"""

df = con.execute(query).df()

print(df.shape)
print(df.head())
print("Reward rate in sample:", df['reward'].mean())

(96898, 7)
  ui_language selected_template  n_eligible  reward  \
0          ar                 E           9       0   
1          en                 F          10       0   
2          en                 G           9       0   
3          es                 L           9       0   
4          id                 B           9       0   

   days_since_last_notification  history_length  time_of_day  
0                     15.073187              29     4.493461  
1                           NaN               2     3.175058  
2                     20.072701              20     6.458808  
3                           NaN               3     6.362917  
4                           NaN               4     2.089792  
Reward rate in sample: 0.13851679085223637


Step 3 — Logistic Regression training

In [19]:
!pip install scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

feature_cols = [
    "selected_template", "ui_language",
    "history_length", "n_eligible",
    "time_of_day", "days_since_last_notification"
]

X = df[feature_cols].copy()
y = df["reward"]

# imputatie (zelfde logica als later met 999)
X["days_since_last_notification"] = X["days_since_last_notification"].fillna(999)

cat_cols = ["ui_language", "selected_template", "time_of_day"]  # time_of_day vaak categorisch
num_cols = ["history_length", "n_eligible", "days_since_last_notification"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("AUC:", auc)

AUC: 0.7700622314562436


c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
model.named_steps["preprocess"].get_feature_names_out()[:20]

array(['cat__ui_language_ar', 'cat__ui_language_cs',
       'cat__ui_language_de', 'cat__ui_language_dn',
       'cat__ui_language_el', 'cat__ui_language_en',
       'cat__ui_language_es', 'cat__ui_language_fr',
       'cat__ui_language_hi', 'cat__ui_language_hu',
       'cat__ui_language_id', 'cat__ui_language_it',
       'cat__ui_language_ja', 'cat__ui_language_ko',
       'cat__ui_language_pl', 'cat__ui_language_pt',
       'cat__ui_language_ro', 'cat__ui_language_ru',
       'cat__ui_language_th', 'cat__ui_language_tr'], dtype=object)

Wat does 0.77 AUC mean concrete?
0.5 = random guessing
0.6 = poor signal
0.7 = solidly predictive
0.8+ = strong model
The model has real predictive signal.
An AUC of ~0.77 means it can rank situations with higher vs lower chance of reward noticeably 
better than random (0.5). So the features/templates correlate with reward.

Stap 4C — Check template ranking according to model

In [27]:
# =========================
# Step 4A + 4B + 5 (ONE cell) — works with your column names + single parquet
# =========================
import duckdb
import numpy as np
import pandas as pd

# ---- PATH (your dataset) ----
PATH = r"C:\Users\Fausto\Downloads\train_rsample_100k.parquet"

# ---- sample size (<= 100k if you want) ----
N_POLICY_ROWS = 100_000

# ---- requires: a trained sklearn pipeline named `model` with predict_proba ----
# model.predict_proba(X)[:,1] must work

con = duckdb.connect(database=":memory:")

# 4A + 4B: sample events, explode eligible_templates, compute candidate-specific recency feature
sql = f"""
WITH base AS (
  SELECT
    row_number() OVER () AS row_id,
    ui_language,
    selected_template AS logged_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,
    history,
    history_length,
    time_of_day
  FROM read_parquet('{PATH}')
  WHERE eligible_templates IS NOT NULL
    AND array_length(eligible_templates) >= 2
),
sampled AS (
  SELECT *
  FROM base
  ORDER BY random()
  LIMIT {N_POLICY_ROWS}
),
long AS (
  SELECT
    s.row_id,
    s.ui_language,
    s.logged_template,
    s.reward,
    s.n_eligible,
    s.history_length,
    s.time_of_day,
    cand AS selected_template,
    COALESCE(
      (
        SELECT min(h.n_days)
        FROM unnest(s.history) AS u(h)
        WHERE h.template = cand
      ),
      999
    ) AS days_since_last_notification
  FROM sampled s
  CROSS JOIN unnest(s.eligible_templates) AS u2(cand)
)
SELECT *
FROM long;
"""

df_long = con.execute(sql).df()

# ---- Score each candidate template with your trained model ----
X_long = df_long[[
    "ui_language",
    "selected_template",
    "n_eligible",
    "history_length",
    "time_of_day",
    "days_since_last_notification"
]].copy()

# Safety (if your model does not impute internally)
X_long["days_since_last_notification"] = X_long["days_since_last_notification"].fillna(999)

df_long["score"] = model.predict_proba(X_long)[:, 1]

# ---- 4B: Greedy choice per event ----
idx = df_long.groupby("row_id")["score"].idxmax()
df_choice = df_long.loc[idx, ["row_id", "selected_template", "score"]].rename(
    columns={"selected_template": "greedy_template", "score": "greedy_score"}
)

# ---- 5: IPS evaluation (assumes uniform logging over eligible_templates) ----
df_event = (
    df_long[["row_id", "ui_language", "logged_template", "reward", "n_eligible", "history_length", "time_of_day"]]
    .drop_duplicates("row_id")
    .merge(df_choice, on="row_id", how="left")
)

df_event["match"] = (df_event["greedy_template"] == df_event["logged_template"]).astype(int)

ips_estimate = np.mean(df_event["match"] * df_event["reward"] * df_event["n_eligible"])
match_rate = df_event["match"].mean()
sanity_E_match_w = np.mean(df_event["match"] * df_event["n_eligible"])  # should be ~1 if uniform logging
matched_reward_mean = df_event.loc[df_event["match"] == 1, "reward"].mean()

print(f"Events evaluated: {len(df_event):,}")
print(f"Match rate (greedy==logged): {match_rate:.4f}")
print(f"Sanity E[match * n_eligible] (~1 if uniform logging): {sanity_E_match_w:.4f}")
print(f"Matched reward mean (NOT unbiased): {matched_reward_mean:.4f}")
print(f"IPS estimate of greedy policy reward: {ips_estimate:.4f}")

df_event.head()

Events evaluated: 96,898
Match rate (greedy==logged): 0.1069
Sanity E[match * n_eligible] (~1 if uniform logging): 0.9965
Matched reward mean (NOT unbiased): 0.1370
IPS estimate of greedy policy reward: 0.1413


,row_id,ui_language,logged_template,reward,n_eligible,history_length,time_of_day,greedy_template,greedy_score,match
0,68313,en,F,0,9,15,8.477662,L,0.115027,0
1,88289,es,A,0,10,23,3.470266,A,0.394603,1
2,62990,fr,J,1,10,6,5.224375,A,0.182068,0
3,96546,es,K,0,9,10,0.628831,L,0.078104,0
4,2106,pt,D,0,9,1,7.833079,L,0.027282,0


This output means the model mainly learns that “Template C is the best on average,” regardless of the situation.

It does not really learn something like: “For user type X, template A works better, but for user type Y, template D works better.”

In other words, it identifies a global winning template, but it does not discover strong differences across contexts (language, history, time of day, etc.), and that’s why there is no additional uplift.